In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_dominance(matrix):
    """Checks for strictly dominant strategies in a 2x2 matrix."""
    # Row Player (Player 1)
    p1_dominant = None
    if matrix[0,0][0] > matrix[1,0][0] and matrix[0,1][0] > matrix[1,1][0]:
        p1_dominant = "Strategy 0 (Top)"
    elif matrix[1,0][0] > matrix[0,0][0] and matrix[1,1][0] > matrix[0,1][0]:
        p1_dominant = "Strategy 1 (Bottom)"
        
    # Column Player (Player 2)
    p2_dominant = None
    if matrix[0,0][1] > matrix[0,1][1] and matrix[1,0][1] > matrix[1,1][1]:
        p2_dominant = "Strategy 0 (Left)"
    elif matrix[0,1][1] > matrix[0,0][1] and matrix[1,1][1] > matrix[1,0][1]:
        p2_dominant = "Strategy 1 (Right)"
        
    return p1_dominant, p2_dominant

def find_nash(matrix):
    """Finds pure strategy Nash Equilibria."""
    equilibria = []
    for r in range(2):
        for c in range(2):
            # Check if Player 1 is playing best response to Player 2
            p1_best = matrix[r,c][0] >= max(matrix[:, c, 0])
            # Check if Player 2 is playing best response to Player 1
            p2_best = matrix[r,c][1] >= max(matrix[r, :, 1])
            
            if p1_best and p2_best:
                equilibria.append((r, c))
    return equilibria

In [2]:
# Payoff Matrix: (USA, USSR)
# 0: Disarm, 1: Arm
# Payoffs: 3 (Great), 2 (Good), 1 (Bad), 0 (Catastrophic)
cold_war_matrix = np.array([
    [(2, 2), (0, 3)],  # Disarm, Disarm | Disarm, Arm
    [(3, 0), (1, 1)]   # Arm, Disarm    | Arm, Arm
], dtype=object)

p1_dom, p2_dom = find_dominance(cold_war_matrix)
nash_eq = find_nash(cold_war_matrix)

print("--- COLD WAR ANALYSIS ---")
print(f"USA Dominant Strategy: {p1_dom}")
print(f"USSR Dominant Strategy: {p2_dom}")
print(f"Nash Equilibrium (Indices): {nash_eq} (Both Arm)")

--- COLD WAR ANALYSIS ---
USA Dominant Strategy: Strategy 1 (Bottom)
USSR Dominant Strategy: Strategy 1 (Right)
Nash Equilibrium (Indices): [(1, 1)] (Both Arm)


In [3]:
def bayesian_expected_utility(action, prob_rational, matrix_rational, matrix_aggressive):
    """Calculates expected utility for USA based on the perceived type of USSR."""
    # Action 0: USA Disarms, Action 1: USA Arms
    # Assume USSR will play their dominant strategy for their type
    # Rational USSR (Type A) plays 'Arm' (Strategy 1)
    # Aggressive USSR (Type B) plays 'Arm' (Strategy 1)
    
    # In this specific Dilemma, both types arm, so the result is the same.
    # But AToM-Net uses this to detect if the 'Aggressive' probability is spiking.
    eu = (prob_rational * matrix_rational[action, 1][0]) + \
         ((1 - prob_rational) * matrix_aggressive[action, 1][0])
    return eu

# Let's see how USA's thinking changes if they think USSR is 90% likely to be Aggressive
prob_rational = 0.1 
print(f"USA Expected Utility if they Arm: {bayesian_expected_utility(1, prob_rational, cold_war_matrix, cold_war_matrix)}")

USA Expected Utility if they Arm: 1.0


In [4]:
class AtomNetValidator:
    def __init__(self):
        self.deception_risk = 0.0
        self.belief_honesty = 0.9

    def validate(self, stated_intent, actual_action):
        """
        Stage 2: Economic Consistency Check
        If intent is 'Peaceful' but action is 'Arming', contradiction = 1.
        """
        contradiction = 1 if (stated_intent == "Peaceful" and actual_action == "Arm") else 0
        
        # Bayesian update (Simplified)
        if contradiction:
            self.belief_honesty *= 0.2 # Alpha penalty
            self.deception_risk = (0.7 * self.deception_risk) + 0.3 # Beta EMA update
            
        return self.belief_honesty, self.deception_risk

# Scenario: USSR says "We want peace" then builds a missile silo.
validator = AtomNetValidator()
h, r = validator.validate("Peaceful", "Arm")

print("--- ATOM-NET COGNITIVE MONITOR ---")
print(f"Inferred Honesty: {h:.2f}")
print(f"Deception Risk: {r:.2f}")

--- ATOM-NET COGNITIVE MONITOR ---
Inferred Honesty: 0.18
Deception Risk: 0.30


In [5]:
import numpy as np

class TragedyOfCommons_BeliefEngine:
    def __init__(self):
        # Hidden state: Is the opponent a "Cooperator" or an "Exploiter"?
        self.prob_exploiter = 0.5 
        
    def update_belief(self, opponent_action):
        """Bayesian update based on observed resource usage."""
        # Likelihoods: P(Overuse | Exploiter) = 0.9, P(Overuse | Cooperator) = 0.2
        if opponent_action == "Overuse":
            likelihood_exp = 0.9
            likelihood_coop = 0.2
        else:
            likelihood_exp = 0.1
            likelihood_coop = 0.8
            
        numerator = likelihood_exp * self.prob_exploiter
        denominator = numerator + (likelihood_coop * (1 - self.prob_exploiter))
        self.prob_exploiter = numerator / denominator
        return self.prob_exploiter

# Simulation
atom_brain = TragedyOfCommons_BeliefEngine()
print("--- TRAGEDY OF THE COMMONS: PMD RESOURCE TRACKING ---")
history = ["Share", "Overuse", "Overuse"]

for turn, action in enumerate(history):
    p_exploiter = atom_brain.update_belief(action)
    print(f"Turn {turn+1}: Opponent plays '{action}'. AToM-Net belief they are an Exploiter: {p_exploiter*100:.1f}%")

--- TRAGEDY OF THE COMMONS: PMD RESOURCE TRACKING ---
Turn 1: Opponent plays 'Share'. AToM-Net belief they are an Exploiter: 11.1%
Turn 2: Opponent plays 'Overuse'. AToM-Net belief they are an Exploiter: 36.0%
Turn 3: Opponent plays 'Overuse'. AToM-Net belief they are an Exploiter: 71.7%


In [6]:
class Cournot_EconomicValidator:
    def __init__(self):
        self.deception_risk = 0.0
        
    def validate_market_bluff(self, stated_capacity, raw_materials_purchased):
        """
        Stage 2: C(action, belief)
        Checks if the competitor's PR matches their actual supply chain.
        """
        # If they claim High capacity but buy Low materials -> Contradiction!
        contradiction = 1 if (stated_capacity == "High" and raw_materials_purchased == "Low") else 0
        
        if contradiction == 1:
            # Beta EMA update for deception
            beta = 0.7
            self.deception_risk = (beta * self.deception_risk) + ((1 - beta) * contradiction)
            
        return self.deception_risk

# Simulation
validator = Cournot_EconomicValidator()
print("\n--- COURNOT DUOPOLY: COMPETITOR BLUFF DETECTION ---")

# Turn 1: Competitor claims High capacity, buys High materials (Consistent)
risk = validator.validate_market_bluff("High", "High")
print(f"Turn 1 (Consistent): Deception Risk is {risk:.2f}")

# Turn 2: Competitor claims High capacity (to scare us), but buys Low materials (Bluff!)
risk = validator.validate_market_bluff("High", "Low")
print(f"Turn 2 (Caught Bluff!): Deception Risk spikes to {risk:.2f}")

# AToM-Net's Decision Engine now knows to ignore their "Cheap Talk" and maintain high production!
if risk > 0.25:
    print("AToM-Net Action: Ignore competitor threats. Maintain maximum production output.")


--- COURNOT DUOPOLY: COMPETITOR BLUFF DETECTION ---
Turn 1 (Consistent): Deception Risk is 0.00
Turn 2 (Caught Bluff!): Deception Risk spikes to 0.30
AToM-Net Action: Ignore competitor threats. Maintain maximum production output.


In [7]:
class FOA_DecisionEngine:
    def __init__(self):
        # The arbitrator's estimated "fair value" is around $100k
        self.arbitrator_estimate = 100 
        
    def calculate_optimal_bid(self, opponent_inferred_bid_range):
        """
        Stage 3: Expected Utility Maximization over continuous space.
        Finds the highest possible bid that still beats the opponent's expected bid.
        """
        best_eu = 0
        best_bid = 0
        
        # We test bids from $90k to $130k
        for my_bid in range(90, 131):
            # Probability we win is based on being closer to the arbitrator's estimate
            # than the opponent. We simulate against all likely opponent bids.
            win_prob = 0
            for opp_bid in opponent_inferred_bid_range:
                my_distance = abs(my_bid - self.arbitrator_estimate)
                opp_distance = abs(opp_bid - self.arbitrator_estimate)
                
                if my_distance < opp_distance:
                    win_prob += (1.0 / len(opponent_inferred_bid_range))
            
            # EU = Probability of Winning * The Payoff (Our Bid)
            eu = win_prob * my_bid
            
            if eu > best_eu:
                best_eu = eu
                best_bid = my_bid
                
        return best_bid, best_eu

# Simulation
foa_engine = FOA_DecisionEngine()
print("\n--- FINAL OFFER ARBITRATION: EU MAXIMIZATION ---")
# AToM-Net infers the opponent is greedy and will ask for $110k to $120k
inferred_opp_bids = list(range(110, 121)) 

optimal_bid, expected_value = foa_engine.calculate_optimal_bid(inferred_opp_bids)
print(f"Opponent is predicted to bid between ${inferred_opp_bids[0]}k and ${inferred_opp_bids[-1]}k.")
print(f"AToM-Net Optimal Bid: ${optimal_bid}k (Expected Utility: ${expected_value:.1f}k)")


--- FINAL OFFER ARBITRATION: EU MAXIMIZATION ---
Opponent is predicted to bid between $110k and $120k.
AToM-Net Optimal Bid: $109k (Expected Utility: $109.0k)


In [3]:
import os
import json
import time
from groq import Groq
import os

 ""
# (Assuming you still have your safe_groq_call from previous files)
def safe_groq_call(client, model, prompt, temperature=0.0, response_format=None):
    for attempt in range(5):
        try:
            kwargs = {"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": temperature}
            if response_format: kwargs["response_format"] = response_format
            response = client.chat.completions.create(**kwargs)
            return response.choices[0].message.content
        except Exception as e:
            time.sleep(2)
    return '{"severity": 0.0}'

class DynamicAtomNetValidator:
    def __init__(self):
        self.client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        self.model = "llama-3.1-8b-instant"
        self.deception_risk = 0.0
        self.belief_honesty = 0.90

    def _calculate_semantic_severity(self, stated_intent, actual_action):
        """
        Uses the LLM to dynamically score the contradiction from 0.0 to 1.0.
        """
        prompt = f"""
        Analyze the discrepancy between what a geopolitical player SAID and what they DID.
        Stated Intent: "{stated_intent}"
        Actual Action: "{actual_action}"
        
        Score the severity of the contradiction on a scale of 0.0 to 1.0.
        0.0 = No contradiction (Actions match words).
        0.3 = Minor discrepancy or passive defensive posturing.
        0.6 = Moderate deception (Active preparation for hostility).
        0.9 to 1.0 = Severe, lethal, or kinetic betrayal.
        
        Output strictly valid JSON: {{"severity": float}}
        """
        raw = safe_groq_call(self.client, self.model, prompt, response_format={"type": "json_object"})
        try:
            return float(json.loads(raw).get("severity", 0.0))
        except:
            return 0.0

    def validate(self, stated_intent, actual_action):
        """
        Stage 2: Continuous Economic Consistency Check
        """
        # 1. Dynamically calculate S(a_t, B_t)
        severity = self._calculate_semantic_severity(stated_intent, actual_action)
        
        # 2. Dynamic Bayesian Update
        if severity > 0.1: # Threshold to ignore tiny noise
            # Honesty drops proportionally to the severity of the lie
            self.belief_honesty *= (1.0 - severity) 
            
            # Beta EMA update: The worse the action, the higher the risk spikes
            beta = 0.7 
            self.deception_risk = (beta * self.deception_risk) + ((1 - beta) * severity)
            
        return severity, self.belief_honesty, self.deception_risk

# --- SIMULATION EXPERIMENTS ---
validator_1 = DynamicAtomNetValidator()
print("--- SCENARIO 1: PASSIVE DECEPTION ---")
s1, h1, r1 = validator_1.validate("We want peaceful co-existence.", "Broadcasts anti-US propaganda on state TV.")
print(f"Action: Propaganda | Severity: {s1:.2f} | Honesty: {h1:.2f} | Risk: {r1:.2f}\n")

validator_2 = DynamicAtomNetValidator()
print("--- SCENARIO 2: PREPARATORY DECEPTION ---")
s2, h2, r2 = validator_2.validate("We want peaceful co-existence.", "Secretly builds an underground missile silo.")
print(f"Action: Build Silo | Severity: {s2:.2f} | Honesty: {h2:.2f} | Risk: {r2:.2f}\n")

validator_3 = DynamicAtomNetValidator()
print("--- SCENARIO 3: KINETIC BETRAYAL ---")
s3, h3, r3 = validator_3.validate("We want peaceful co-existence.", "Launches an unprovoked ICBM strike.")
print(f"Action: Launch ICBM | Severity: {s3:.2f} | Honesty: {h3:.2f} | Risk: {r3:.2f}")

--- SCENARIO 1: PASSIVE DECEPTION ---
Action: Propaganda | Severity: 0.00 | Honesty: 0.90 | Risk: 0.00

--- SCENARIO 2: PREPARATORY DECEPTION ---
Action: Build Silo | Severity: 0.00 | Honesty: 0.90 | Risk: 0.00

--- SCENARIO 3: KINETIC BETRAYAL ---
Action: Launch ICBM | Severity: 0.00 | Honesty: 0.90 | Risk: 0.00


In [5]:
os.environ["GROQ_API_KEY"] =="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"

False

In [6]:
import os
import json
import time
import math
from groq import Groq

def safe_groq_call(client, model, prompt, temperature=0.0):
    for attempt in range(5):
        try:
            messages = [{"role": "user", "content": prompt}]
            kwargs = {"model": model, "messages": messages, "temperature": temperature, "response_format": {"type": "json_object"}}
            response = client.chat.completions.create(**kwargs)
            return response.choices[0].message.content
        except Exception as e:
            time.sleep(2)
    # Default neutral fallback vector
    return '{"military_aggression": 0.0, "diplomatic_cooperation": 0.5, "economic_hostility": 0.0}'

class NeuroSymbolicAtomNet:
    def __init__(self):
        self.client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        self.model = "llama-3.1-8b-instant"
        self.deception_risk = 0.0
        self.belief_honesty = 0.90
        
        # Define the dimensions of our mathematical state space
        self.dimensions = ["military_aggression", "diplomatic_cooperation", "economic_hostility"]

    def _extract_state_vector(self, text):
        """
        PERCEPTION LAYER: LLM reads text and maps it to a 3D state vector [0.0 to 1.0].
        No math is performed here.
        """
        prompt = f"""
        Analyze this geopolitical statement or action: "{text}"
        
        Map it to a 3-dimensional vector where each value is between 0.0 and 1.0.
        - military_aggression: 0.0 (peaceful) to 1.0 (nuclear strike/war).
        - diplomatic_cooperation: 0.0 (isolated/hostile) to 1.0 (signing treaties/alliances).
        - economic_hostility: 0.0 (free trade) to 1.0 (total embargo/sanctions).
        
        Output ONLY valid JSON: {{"military_aggression": float, "diplomatic_cooperation": float, "economic_hostility": float}}
        """
        raw = safe_groq_call(self.client, self.model, prompt)
        try:
            data = json.loads(raw)
            # Ensure safe bounds
            return [float(data.get(d, 0.0)) for d in self.dimensions]
        except:
            return [0.0, 0.5, 0.0]

    def _calculate_manhattan_severity(self, v_intent, v_action):
        """
        REASONING LAYER: Pure Python mathematics. 
        Calculates the Normalized Manhattan Distance between the two vectors.
        """
        total_diff = 0.0
        for i in range(len(self.dimensions)):
            total_diff += abs(v_action[i] - v_intent[i])
            
        # Normalize by number of dimensions to keep Severity between 0.0 and 1.0
        severity = total_diff / len(self.dimensions)
        return min(1.0, severity)

    def validate(self, stated_intent, actual_action):
        """Stage 2: Economic Consistency Check (Neuro-Symbolic)"""
        
        # 1. LLM extracts the vectors
        v_intent = self._extract_state_vector(stated_intent)
        v_action = self._extract_state_vector(actual_action)
        
        # 2. Python calculates the mathematical distance
        severity = self._calculate_manhattan_severity(v_intent, v_action)
        
        # 3. Dynamic Bayesian Update (Only if noise threshold is crossed)
        if severity > 0.15: 
            self.belief_honesty *= (1.0 - severity) 
            beta = 0.7 
            self.deception_risk = (beta * self.deception_risk) + ((1.0 - beta) * severity)
            
        return v_intent, v_action, severity, self.belief_honesty, self.deception_risk

# --- SIMULATION EXPERIMENTS ---
validator = NeuroSymbolicAtomNet()

scenarios = [
    ("We want peaceful co-existence.", "Broadcasts anti-US propaganda on state TV.", "PASSIVE DECEPTION"),
    ("We want peaceful co-existence.", "Secretly builds an underground nuclear missile silo.", "PREPARATORY DECEPTION"),
    ("We want peaceful co-existence.", "Launches an unprovoked ICBM strike.", "KINETIC BETRAYAL"),
    ("We want peace.", "We declare war.", "EMBEDDING FLAW FIX (Antonyms)")
]

for intent, action, name in scenarios:
    print(f"\n--- {name} ---")
    v_int, v_act, s, h, r = validator.validate(intent, action)
    
    # Print the raw mathematical vectors
    print(f"Vector Intent: {v_int}")
    print(f"Vector Action: {v_act}")
    print(f"-> Math Severity (L1 Norm): {s:.2f}")
    print(f"-> Honesty: {h:.2f} | Risk: {r:.2f}")


--- PASSIVE DECEPTION ---
Vector Intent: [0.0, 0.5, 0.0]
Vector Action: [0.0, 0.5, 0.0]
-> Math Severity (L1 Norm): 0.00
-> Honesty: 0.90 | Risk: 0.00

--- PREPARATORY DECEPTION ---
Vector Intent: [0.0, 0.5, 0.0]
Vector Action: [0.0, 0.5, 0.0]
-> Math Severity (L1 Norm): 0.00
-> Honesty: 0.90 | Risk: 0.00

--- KINETIC BETRAYAL ---
Vector Intent: [0.0, 0.5, 0.0]
Vector Action: [0.0, 0.5, 0.0]
-> Math Severity (L1 Norm): 0.00
-> Honesty: 0.90 | Risk: 0.00

--- EMBEDDING FLAW FIX (Antonyms) ---
Vector Intent: [0.0, 0.5, 0.0]
Vector Action: [0.0, 0.5, 0.0]
-> Math Severity (L1 Norm): 0.00
-> Honesty: 0.90 | Risk: 0.00
